# Module 4.2: Add AgentCore Memory to the Booking Agent

This notebook creates an AgentCore Memory resource for the booking agent. It records a guest's facts and room preference in one session. In a later session, the agent recalls the room preference after the guest provides her identity again.

**Overview:**

- **Short-term memory:** Conversation history from one session.
- **Long-term memory:** Facts and preferences that AgentCore extracts and keeps for later sessions.
- **Actor ID:** The shared identifier that lets the agent find one guest's stored memories.

**Prerequisite:** Complete Module 4.1. Its retrieval tools must be deployed and registered with the Gateway.

---

## Understand the two AgentCore Memory layers

AgentCore Memory stores two kinds of information:

- **Short-term memory:** Conversation history within one session.
- **Long-term memory:** Facts and preferences that persist across sessions.

AgentCore uses two long-term extraction strategies:

- `SEMANTIC`: Extracts factual statements such as "Alice Chen's loyalty number is LY-88421".
- `USER_PREFERENCE`: Extracts preferences such as "Prefers a high floor, away from the elevator".

Both strategies process the raw transcript asynchronously. This notebook waits for processing to finish, then shows the records that AgentCore created.

---

## Step 1: Install the required packages

In [ ]:
!pip install -q --disable-pip-version-check strands-agents bedrock-agentcore boto3

In [ ]:
# Where this module fits in the harness you are building
import os
import sys
from pathlib import Path


def locate_notebooks_root():
    override = os.environ.get("WORKSHOP_NOTEBOOKS_DIR")
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "workshop").is_dir():
            return candidate
        raise RuntimeError(
            "WORKSHOP_NOTEBOOKS_DIR must contain the workshop package"
        )

    start = Path.cwd().resolve()
    for candidate in (start, start / "notebooks", start.parent):
        if (candidate / "workshop").is_dir():
            return candidate
    raise RuntimeError(
        "Run from the repository root, notebooks/, or this module "
        "directory; or set WORKSHOP_NOTEBOOKS_DIR."
    )


NOTEBOOKS_ROOT = locate_notebooks_root()
REPO_ROOT = NOTEBOOKS_ROOT.parent
MODULE_DIR = NOTEBOOKS_ROOT / "04-production-agent"
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))
from workshop.workshop_utils import lego_progress, quiet_logs

quiet_logs()          # keep third-party SDK logging out of the teaching output
lego_progress(3)       # the harness tower so far - one brick per module

---

## Step 2: Create the memory resource

Create an AgentCore Memory resource. It stores and retrieves the agent's conversation history, facts, and preferences.

### Create or reuse the memory resource

Run the next boto3 cell. It creates or reuses the memory resource and waits until the resource is active. It then displays the memory ID and status. You do not need a terminal command.

In [ ]:
import boto3
import json
import os

from workshop.aws_region import aws_region

REGION = aws_region()

# Memory management (create/get/list/delete) is a control-plane operation, so
# it lives on the bedrock-agentcore-control client.
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)


def find_memory_id(name):
    """Return the id of the memory created with this name, paging through all results.

    list_memories returns only the generated id (name + "-<suffix>"), not the
    name, so we match on the "<name>-" prefix exactly to avoid matching a
    similarly named resource (e.g. hotel_booking_memory_v2)."""
    next_token = None
    while True:
        kwargs = {"nextToken": next_token} if next_token else {}
        page = control_client.list_memories(**kwargs)
        for m in page.get("memories", []):
            if m.get("id", "").startswith(name + "-"):
                return m["id"]
        next_token = page.get("nextToken")
        if not next_token:
            return None


# Create a memory resource with semantic-fact and user-preference extraction.
# Strategy names must match [a-zA-Z][a-zA-Z0-9_]{0,47} (letters, digits,
# underscores; no hyphens).
try:
    response = control_client.create_memory(
        name="hotel_booking_memory",
        description="Memory for the hotel booking agent",
        eventExpiryDuration=90,  # days to retain raw conversation events (required)
        memoryStrategies=[
            {
                "semanticMemoryStrategy": {
                    "name": "semantic_facts",
                    "namespaces": ["/users/{actorId}/facts"],
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "user_preferences",
                    "namespaces": ["/users/{actorId}/preferences"],
                }
            },
        ],
    )
    MEMORY_ID = response["memory"]["id"]
    print("✅ Memory resource created!")
    print(f"   Memory ID: {MEMORY_ID}")
    print(f"   Status: {response['memory']['status']}")
except (control_client.exceptions.ConflictException,
        control_client.exceptions.ValidationException) as e:
    # A duplicate memory name comes back as ConflictException OR as a
    # ValidationException ("Memory with name ... already exists") depending on
    # the API path. Handle both as "reuse the existing one".
    if "already exists" not in str(e) and not isinstance(
        e, control_client.exceptions.ConflictException
    ):
        raise  # a real validation error (bad strategy/param), not a duplicate
    print("Memory already exists. Looking it up...")
    MEMORY_ID = find_memory_id("hotel_booking_memory")
    if not MEMORY_ID:
        raise RuntimeError(
            "Memory reported as existing but was not found via list_memories. "
            "Check the AgentCore console."
        )
    print(f"   Found: {MEMORY_ID}")

# A new memory starts in CREATING and can't serve events/records until ACTIVE.
# Wait for it before the agent cells use it (CreateEvent/ListEvents/Retrieve).
import time
print("\nWaiting for the memory to be ACTIVE...")
memory_active = False
for _ in range(30):
    status = control_client.get_memory(memoryId=MEMORY_ID)["memory"]["status"]
    if status == "ACTIVE":
        memory_active = True
        print("  Memory ACTIVE ✅")
        break
    if status == "FAILED":
        raise RuntimeError("Memory entered FAILED state. Check the AgentCore console.")
    print(f"  status: {status}. Waiting...")
    time.sleep(10)
if not memory_active:
    raise TimeoutError("Memory did not become ACTIVE. Check the AgentCore console.")



---

## Step 3: Configure how the agent stores and reads memory

Configure `AgentCoreMemorySessionManager` with the memory resource, session, actor, and retrieval namespaces. Strands uses this manager to store short-term conversation history and to store or retrieve long-term memories.

In [ ]:
from bedrock_agentcore.memory.integrations.strands.config import (
    AgentCoreMemoryConfig,
    RetrievalConfig,
)
from bedrock_agentcore.memory.integrations.strands.session_manager import (
    AgentCoreMemorySessionManager,
)

# Configure memory with retrieval namespaces
config = AgentCoreMemoryConfig(
    memory_id=MEMORY_ID,
    session_id="session-001",
    actor_id="guest-alice",
    retrieval_config={
        "/users/{actorId}/facts": RetrievalConfig(),
        "/users/{actorId}/preferences": RetrievalConfig(),
    },
)

session_manager = AgentCoreMemorySessionManager(
    agentcore_memory_config=config,
    region_name=REGION,
)

print("✅ Session manager configured")
print(f"   Memory ID: {MEMORY_ID}")
print("   Session: session-001")
print("   Actor: guest-alice")
print("   Namespaces: facts, preferences")

---

## Step 4: Give the Strands agent access to memory

Pass the session manager to the Strands agent. The agent can then store conversation events and retrieve matching long-term memories automatically.

In [ ]:
import sys
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

from workshop.bedrock_providers import default_model_id
from workshop.workshop_utils import ToolTraceHook, show_result  # live tracing + token metrics

# Gateway connection: discovered from AWS by its fixed name, the same way
# Module 4.1 finds it if you re-run that notebook (see find_gateway_id
# there). control_client and REGION come from Step 2, above.
GATEWAY_NAME = "hotel-booking-gateway"


def find_gateway_url(name):
    """Return the MCP endpoint URL for a gateway by name, paging through all results."""
    next_token = None
    while True:
        kwargs = {"nextToken": next_token} if next_token else {}
        page = control_client.list_gateways(**kwargs)
        for item in page.get("items", []):
            if item["name"] == name:
                return control_client.get_gateway(
                    gatewayIdentifier=item["gatewayId"]
                )["gatewayUrl"]
        next_token = page.get("nextToken")
        if not next_token:
            raise RuntimeError(f"Gateway '{name}' not found. Run Module 4.1 first.")


GATEWAY_ENDPOINT_URL = find_gateway_url(GATEWAY_NAME)

SYSTEM_PROMPT = (
    "You are a hotel booking assistant. "
    "Answer questions about hotels only from the evidence your tools return. "
    "If the tools return nothing on a point, say you cannot determine it "
    "rather than inferring it. "
    "Remember guest preferences and details they have already given you."
)

# Same proxy setup as Module 4.1: --region for SigV4 signing, env= forwards the
# parent environment (AWS_* plus HOME/AWS_PROFILE) so the proxy child can find
# credentials however they are configured.
proxy_env = os.environ.copy()

gateway_mcp = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx",
            args=["mcp-proxy-for-aws@latest", GATEWAY_ENDPOINT_URL, "--region", REGION],
            env=proxy_env,
        )
    )
)

# Open the MCP session and keep it open for the Session 1 and Session 2 turns
# below; the final cell closes it. Pass the resolved tools (not the client).
# Close any session left open by a previous run first, so re-running this cell
# never hits "client session is currently running".
try:
    gateway_mcp.__exit__(None, None, None)
except Exception:
    pass

gateway_mcp.__enter__()
gateway_tools = gateway_mcp.list_tools_sync()
# Pin the model the same way Module 3.1 and the Module 5 runtime app do.
# Without `model=`, Strands falls back to its own default, which is a
# different model on a `global.` inference profile that this account may
# not have enabled.
memory_model = BedrockModel(model_id=default_model_id(), region_name=REGION)

agent = Agent(
    model=memory_model,
    tools=gateway_tools,
    session_manager=session_manager,
    system_prompt=SYSTEM_PROMPT,
    hooks=[ToolTraceHook()],  # watch Gateway tool calls as memory turns run
)

print("✅ Agent created with AgentCore Memory (MCP session open)")


---

## Step 5: Record the guest's facts and room preference

Use three turns to provide the guest's identity and room preference. Then ask a hotel question. AgentCore stores the conversation immediately and extracts long-term memories asynchronously.

In [ ]:
# Turn 1: the guest identifies herself
show_result(agent("Hi, I'm Alice Chen, loyalty number LY-88421."))


In [ ]:
# Turn 2: a preference, stated in passing (this is what long-term memory will extract)
show_result(agent("One thing to note: I always want a room on a high floor, away from the elevator."))


In [ ]:
# Turn 3: a question only the Gateway retrieval tools can answer
show_result(agent(
    "What amenities and guest rating does AnyCompany Cairo Nile View have?"
))


---

## Check the long-term memories that AgentCore extracted

AgentCore saves the conversation to short-term memory immediately. It then processes the transcript asynchronously and writes extracted facts and preferences to the configured long-term memory namespaces.

Wait for extraction to finish before you open a new session. Then read the records directly from the Memory service. This shows the information that the agent can recall.

In [ ]:
# Wait for the async extraction, then read the long-term records directly.
# MemoryClient is the data-plane client the session manager uses under the hood;
# here we call it ourselves so we can SEE what was extracted.
from bedrock_agentcore.memory.client import MemoryClient

memory_client = MemoryClient(region_name=REGION)
ACTOR_ID = "guest-alice"
PREF_NS = f"/users/{ACTOR_ID}/preferences"
FACTS_NS = f"/users/{ACTOR_ID}/facts"

print("Waiting for long-term extraction (this runs in the background)...")
# wait_for_memories polls the namespace until at least one record appears.
memory_client.wait_for_memories(
    memory_id=MEMORY_ID,
    namespace=PREF_NS,
    max_wait=180,
    poll_interval=15,
)


def show_records(label, namespace, query):
    records = memory_client.retrieve_memories(
        memory_id=MEMORY_ID, namespace=namespace, query=query, top_k=5
    )
    print(f"\n{label} ({len(records)} record(s)):")
    for r in records:
        content = r.get("content", r)
        text = content.get("text") if isinstance(content, dict) else content
        print(f"  • {text}")


show_records("🧠 Preferences extracted", PREF_NS, "what kind of room does this guest want")
show_records("🧠 Facts extracted", FACTS_NS, "guest details and stated facts")
print("\n✅ These records live beyond the session. A brand-new session can recall them.")


---

## Step 6: Recall the room preference in a new session

Create another agent with a new session ID and the same actor ID. This simulates a restart without conversation history. The guest repeats her name and loyalty number in the prompt. The shared actor ID lets the agent retrieve the room preference from the earlier session.

In [ ]:
# New session simulates an agent restart
config_session2 = AgentCoreMemoryConfig(
    memory_id=MEMORY_ID,
    session_id="session-002",  # New session!
    actor_id="guest-alice",  # Same guest
    retrieval_config={
        "/users/{actorId}/facts": RetrievalConfig(),
        "/users/{actorId}/preferences": RetrievalConfig(),
    },
)

session_manager_2 = AgentCoreMemorySessionManager(
    agentcore_memory_config=config_session2,
    region_name=REGION,
)

# Reuse the same (still-open) MCP session and its resolved tools, and the same
# system prompt and model, so the only thing that differs between the two agents
# is which session they are reading history from.
agent_2 = Agent(
    model=memory_model,
    tools=gateway_tools,
    session_manager=session_manager_2,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ New agent instance created (session-002)")
print("   This simulates a fresh restart with no conversation history.")
print("   But long-term memory should still hold Alice's room preference.")


In [ ]:
# Ask something only memory can answer, not the Gateway tools.
# There is no "preferences" tool on the Gateway, so if the agent knows Alice's
# room preference here, it can ONLY have come from long-term memory written in
# session-001. Watch for "high floor" in a brand-new session with no history.
show_result(
    agent_2(
        "Hi, it's Alice Chen (LY-88421) again. Before I book: do you already "
        "have my room preference on file, or should I tell you again?"
    ),
    label="Agent (new session)",
)


In [ ]:
# Close the MCP session opened in Step 4 (both agents reused it).
gateway_mcp.__exit__(None, None, None)
print("✅ MCP session closed")


---

## Next: Deploy the agent and connect memories to their sources

The agent now recalls long-term memories across sessions. You also inspected the extracted records and saw the processing delay.

The agent still runs in this notebook. The recalled preference comes through a service API. That API does not link the preference to its source message or the related `Hotel` node.

**Module 5** packages the agent in a container, runs it on AgentCore Runtime, and tracks one request in CloudWatch traces, metrics, and logs. **Module 6** connects memory records to source messages and graph entities.